In [5]:

!pip install -q langchain-openai langchain-core
import os
from getpass import getpass
os.environ["API_KEY"] = getpass("Enter your API key: ")


Enter your API key: ··········


In [6]:

import json
from dataclasses import dataclass
from typing import TypedDict
from langchain_openai import ChatOpenAI
import pandas as pd

#Defining schema

SCHEMA_DEF = {
    "STUDYID": "Unique study identifier.",
    "DOMAIN": "Domain abbreviation, always 'AE' for the Adverse Events domain.",
    "USUBJID": "Unique subject identifier for a participant in the study.",
    "AESEQ": "Sequence number given to ensure uniqueness of records within a subject.",
    "AESPID": "Sponsor-defined identifier for the adverse event record.",
    "AETERM": (
        "The site reported term for the adverse event(e.g., "
        "'HEADACHE', 'NAUSEA', 'ATRIAL FIBRILLATION'). Use this column when "
        "the user asks about a specific condition, symptom, or diagnosis as "
        "the investigator originally wrote it."
    ),
    "AEDECOD": (
        "The MedDRA-coded preferred term (PT) for the adverse event. Similar "
        "in meaning to AETERM but standardized/coded terminology."
    ),
    "AELLT": "MedDRA Lowest Level Term- the most granular level of MedDRA coding.",
    "AEHLT": "MedDRA High Level Term- groups related preferred terms together.",
    "AEHLGT": "MedDRA High Level Group Term- a broader grouping above AEHLT.",
    "AEBODSYS": "Body system or organ class as coded by the sponsor, similar to AESOC.",
    "AESOC": (
        "The MedDRA System Organ Class (body system) the adverse event "
        "belongs to (e.g., 'CARDIAC DISORDERS', 'SKIN AND SUBCUTANEOUS "
        "TISSUE DISORDERS', 'NERVOUS SYSTEM DISORDERS'). Use this column "
        "when the user asks about a body system, organ class, or general "
        "category of adverse events rather than one specific event."
    ),
    "AESEV": (
        "The severity/intensity of the adverse event. Valid values are "
        "'MILD', 'MODERATE', 'SEVERE'. Use this column when the user asks "
        "about severity, intensity, or how bad/serious an event was."
    ),
    "AESER": (
        "Whether the adverse event was a serious adverse event (SAE). "
        "Valid values are typically 'Y'/'N'. Use this column when the user "
        "asks whether events were 'serious' in the regulatory sense. Seriousness does not imply severity."
    ),
    "AEACN": (
        "Action taken with study treatment as a result of the adverse event "
        "(e.g., 'DOSE NOT CHANGED', 'DRUG WITHDRAWN', 'DOSE REDUCED'). Use "
        "this column when the user asks what happened to a subject's "
        "treatment/dosing because of an event."
    ),
    "AEREL": (
        "Causality/relationship of the adverse event to study treatment, "
        "as assessed by the investigator (e.g., 'NONE', 'POSSIBLE', "
        "'PROBABLE','REMOTE'). Use this column when the user asks whether an "
        "event was caused by or related to the study drug."
    ),
    "AEOUT": (
        "Outcome of the adverse event (e.g., 'RECOVERED/RESOLVED', "
        "'NOT RECOVERED/NOT RESOLVED', 'FATAL'). "
        "Use this column when the user asks what ultimately happened to the "
        "event or patient (resolved, ongoing, fatal, etc.)."
    ),
    "AESCAN": "Whether the event involved cancer('Y'/'N').",
    "AESCONG": "Whether the event involved a congenital anomaly or birth defect ('Y'/'N').",
    "AESDISAB": "Whether the event resulted in persistent or significant disability/incapacity ('Y'/'N').",
    "AESDTH": "Whether the event resulted in death ('Y'/'N').",
    "AESHOSP": "Whether the event required or prolonged hospitalization ('Y'/'N').",
    "AESLIFE": "Whether the event was life-threatening ('Y'/'N').",
    "AESOD": "Whether the event occurred with overdose ('Y'/'N').",
    "AEDTC": (
    "The date/time collected for the adverse event in ISO 8601 format. "
    "Use this column when referring to the recorded date/time captured on the case report form. "
    ),
    "AESTDTC": (
    "The start date/time of the adverse event in ISO 8601 format. "
    "Use this column when the user asks when an adverse event began "
    "or when filtering or analyzing adverse events by their onset date."
    ),
    "AEENDTC": "End date/time of the adverse event, in ISO 8601 format.",
    "AESTDY": "Study day of onset of the adverse event, relative to a reference start date.",
    "AEENDY": "Study day of end of the adverse event, relative to a reference start date.",
}


def build_ae_schema(df: pd.DataFrame) -> dict:
    schema = {}
    for column_name in df.columns:
        if column_name in SCHEMA_DEF:
            schema[column_name] = SCHEMA_DEF[column_name]
        else:
            schema[column_name] = (
                f"Column '{column_name}' from the AE dataset "
                "(no standard description available)."
            )
    return schema

# the following class invokes the LLM and parses user's question to create a JSON output
class ClinicalTrialDataAgent:

    def __init__(self, df: pd.DataFrame, model: str = "openai/gpt-4o-mini"):
        self.df = df
        self.schema = build_ae_schema(df)
        self.llm = ChatOpenAI(
            model=model,
            base_url="https://openrouter.ai/api/v1", #used OpenRouter API as it is free
            api_key=os.environ["API_KEY"],
            temperature=0,
        )

#asks LLM question and passes instructions for processing, returns JSON containing information on target column and filter value
    def parse(self, question: str) -> dict:

        prompt = f"""
The dataframe has these columns (name: description):
{json.dumps(self.schema)}

Read the question below and identify:
- target_column: which column should be filtered (must be one of the columns above)
- filter_value: what value to search for in that column

Question: "{question}"

Respond with ONLY valid JSON in this exact format, nothing else:
{{"target_column": "...", "filter_value": "..."}}
"""
        response = self.llm.invoke(prompt)
        return json.loads(response.content.strip())


    def ask(self, question: str) -> dict:
        parsed = self.parse(question)
        return filter_ae_data(self.df, parsed)

#function returns count of subjects that match the query and a list of matching IDs.
def filter_ae_data(df: pd.DataFrame, parsed_output: dict) -> dict:
    column = parsed_output["target_column"]
    value = parsed_output["filter_value"]
    matches = df[df[column].astype(str).str.contains(value, case=False, na=False)]

    subject_ids = sorted(matches["USUBJID"].unique())

    return {
        "subject_count": len(subject_ids),
        "subject_ids": subject_ids,
    }


In [7]:

df = pd.read_csv("/content/adae.csv")   #load ADAE dataset here

agent = ClinicalTrialDataAgent(df)

#Test Script block: tries 3 sample queries and prints the count of unique subjects and the list of matching IDs that meet the user's criteria.
result = agent.ask("How many patients reported abdominal pain as an adverse event?")
print(result)
result2= agent.ask("How many subjects had a serious event?")
print(result2)
result3= agent.ask("Give me patients who have recovered from the adverse event.")
print(result3)



{'subject_count': 5, 'subject_ids': ['01-705-1393', '01-708-1286', '01-709-1020', '01-713-1209', '01-716-1311']}
{'subject_count': 3, 'subject_ids': ['01-709-1424', '01-718-1170', '01-718-1371']}
{'subject_count': 161, 'subject_ids': ['01-701-1015', '01-701-1023', '01-701-1047', '01-701-1097', '01-701-1111', '01-701-1115', '01-701-1130', '01-701-1146', '01-701-1148', '01-701-1153', '01-701-1180', '01-701-1181', '01-701-1192', '01-701-1203', '01-701-1211', '01-701-1239', '01-701-1275', '01-701-1287', '01-701-1294', '01-701-1302', '01-701-1324', '01-701-1341', '01-701-1363', '01-701-1383', '01-701-1387', '01-701-1392', '01-701-1415', '01-701-1444', '01-702-1082', '01-703-1042', '01-703-1076', '01-703-1086', '01-703-1100', '01-703-1258', '01-703-1299', '01-703-1335', '01-703-1403', '01-704-1008', '01-704-1009', '01-704-1010', '01-704-1017', '01-704-1025', '01-704-1065', '01-704-1093', '01-704-1114', '01-704-1120', '01-704-1164', '01-704-1266', '01-704-1323', '01-704-1332', '01-704-1351', 